## makemore: becoming a backprop ninja

In [ ]:
# there no change change in the first several cells from last lecture

In [ ]:
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [ ]:
# download the names.txt file from github
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-09-16 23:13:54--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt.1’

names.txt.1         100%[===================>] 222.80K  --.-KB/s    in 0.03s   

2026-09-16 23:13:54 (8.64 MB/s) - ‘names.txt.1’ saved [228145/228145]



In [ ]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [ ]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = jnp.array(X)
  Y = jnp.array(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

(182625, 3) (182625,)
(22655, 3) (22655,)
(22866, 3) (22866,)


In [ ]:
# ok biolerplate done, now we get to the action:

In [ ]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = jnp.all(jnp.equal(dt, t))
  app = jnp.allclose(dt, t)
  maxdiff = jnp.max(jnp.abs(dt - t)).item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [ ]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = jax.random.key(2147483647) # for reproducibility
g, k1 = jax.random.split(g)
C  = jax.random.normal(k1, (vocab_size, n_embd))
# Layer 1
g, k2, k3 = jax.random.split(g, 3)
W1 = jax.random.normal(k2, (n_embd * block_size, n_hidden)) * (5/3)/((n_embd * block_size)**0.5)
b1 = jax.random.normal(k3, (n_hidden,)) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
g, k4, k5 = jax.random.split(g, 3)
W2 = jax.random.normal(k4, (n_hidden, vocab_size)) * 0.1
b2 = jax.random.normal(k5, vocab_size) * 0.1
# BatchNorm parameters
g, k6, k7 = jax.random.split(g, 3)
bngain = jax.random.normal(k6, (1, n_hidden))*0.1 + 1.0
bnbias = jax.random.normal(k7, (1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.size for p in parameters)) # number of parameters in total

4137


In [ ]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
g, k8 = jax.random.split(g)
ix = jax.random.randint(k8, (batch_size,), 0, Xtr.shape[0])
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [ ]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb]

embcat = emb.reshape(emb.shape[0], -1)

hprebn = embcat @ W1 + b1

# BatchNorm layer
bmeani = hprebn.mean(0, keepdims=True)
bndiff = hprebn - bmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdims=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Non-linearity
h = jax.nn.tanh(hpreact)

# Linear layer 2
logits = h @ W2 + b2

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdims=True)
norm_logits = logits - logit_maxes
counts = jnp.exp(norm_logits)
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = 1.0 / counts_sum

probs = counts * counts_sum_inv
logprobs = jnp.log(probs)
loss = -logprobs[jnp.arange(n), Yb].mean()

# Jax backward pass
loss

Array(3.4020648, dtype=float32)

In [ ]:
def add_value_grad(intermediates, name, value, grad):
  i = intermediates.setdefault(name, {})
  i["value"] = value
  # i.setdefault("grad", 0)
  # i["grad"] += grad
  i["grad"] = grad
  intermediates[name] = i
  return intermediates

In [ ]:
def loss_fn(logprobs, Yb):
  jax.debug.print("{}", "In loss")
  return -logprobs[jnp.arange(n), Yb].mean(), {}

def logprobs_fn(probs, Yb):
  jax.debug.print("{}", "In logprobs")
  logprobs = jnp.log(probs)
  (loss, intermediates), logprobs_grad = jax.value_and_grad(loss_fn, has_aux=True)(logprobs, Yb)
  intermediates = add_value_grad(intermediates, "logprobs", logprobs, logprobs_grad)
  return loss, intermediates

def probs_fn(counts, counts_sum_inv, Yb):
  jax.debug.print("{}", "In probs")
  probs = counts * counts_sum_inv
  (loss, intermediates), probs_grad = jax.value_and_grad(logprobs_fn, has_aux=True)(probs, Yb)
  intermediates = add_value_grad(intermediates, "probs", probs, probs_grad)
  return loss, intermediates

def counts_sum_fn(counts, Yb):
  jax.debug.print("{}", "In counts_sum")
  counts_sum = counts.sum(1, keepdims=True)

  def counts_sum_inv_fn(counts_sum, Yb):
    jax.debug.print("{}", "In counts_sum_inv")
    counts_sum_inv = 1.0 / counts_sum
    (loss, intermediates), (counts_grad, counts_sum_inv_grad) = jax.value_and_grad(probs_fn, has_aux=True, argnums=(0,1))(counts, counts_sum_inv, Yb)
    intermediates = add_value_grad(intermediates, "counts_sum_inv", counts_sum_inv, counts_sum_inv_grad)
    intermediates = add_value_grad(intermediates, "counts", counts, counts_grad)
    return loss, intermediates

  (loss, intermediates), counts_sum_grad = jax.value_and_grad(counts_sum_inv_fn, has_aux=True)(counts_sum, Yb)
  intermediates = add_value_grad(intermediates, "counts_sum", counts_sum, counts_sum_grad)
  return loss, intermediates

def counts_fn(norm_logits, Yb):
  jax.debug.print("{}", "In counts")
  counts = jnp.exp(norm_logits)
  (loss, intermediates), counts_grad = jax.value_and_grad(counts_sum_fn, has_aux=True)(counts, Yb)
  intermediates = add_value_grad(intermediates, "counts", counts, counts_grad)
  return loss, intermediates

def logit_maxes_fn(logits, Yb):
  jax.debug.print("{}", "In logit_maxes")
  logit_maxes = logits.max(1, keepdims=True)

  def norm_logits_fn(logits, logit_maxes, Yb):
    jax.debug.print("{}", "In norm_logits")
    norm_logits = logits - logit_maxes
    (loss, intermediates), norm_logits_grad = jax.value_and_grad(counts_fn, has_aux=True)(norm_logits, Yb)
    intermediates = add_value_grad(intermediates, "norm_logits", norm_logits, norm_logits_grad)
    return loss, intermediates

  (loss, intermediates), (logits_grad, logit_maxes_grad) = jax.value_and_grad(norm_logits_fn, has_aux=True, argnums=(0,1))(logits, logit_maxes, Yb)
  intermediates = add_value_grad(intermediates, "logit_maxes", logit_maxes, logit_maxes_grad)
  intermediates = add_value_grad(intermediates, "logits", logits, logits_grad)
  return loss, intermediates

def logits_fn(h, W2, b2, Yb):
  jax.debug.print("{}", "In logits")
  logits = h @ W2 + b2
  (loss, intermediates), logits_grad = jax.value_and_grad(logit_maxes_fn, has_aux=True)(logits, Yb)
  intermediates = add_value_grad(intermediates, "logits", logits, logits_grad)
  return loss, intermediates

def h_fn(hpreact, Yb):
  jax.debug.print("{}", "In h")
  h = jax.nn.tanh(hpreact)
  (loss, intermediates), (h_grad, W2_grad, b2_grad) = jax.value_and_grad(logits_fn, has_aux=True, argnums=(0,1,2))(h, W2, b2, Yb)
  intermediates = add_value_grad(intermediates, "h", h, h_grad)
  intermediates = add_value_grad(intermediates, "W2", W2, W2_grad)
  intermediates = add_value_grad(intermediates, "b2", b2, b2_grad)
  return loss, intermediates

def hpreact_fn(bnraw, bngain, bnbias, Yb):
  jax.debug.print("{}", "In hpreact")
  hpreact = bngain * bnraw + bnbias
  (loss, intermediates), hpreact_grad = jax.value_and_grad(h_fn, has_aux=True)(hpreact, Yb)
  intermediates = add_value_grad(intermediates, "hpreact", hpreact, hpreact_grad)
  return loss, intermediates

def bnraw_fn(bndiff, bnvar_inv, Yb):
  jax.debug.print("{}", "In bnraw")
  bnraw = bndiff * bnvar_inv
  (loss, intermediates), (bnraw_grad, bngain_grad, bnbias_grad) = jax.value_and_grad(hpreact_fn, has_aux=True, argnums=(0,1,2))(bnraw, bngain, bnbias, Yb)
  intermediates = add_value_grad(intermediates, "bnraw", bnraw, bnraw_grad)
  intermediates = add_value_grad(intermediates, "bngain", bngain, bngain_grad)
  intermediates = add_value_grad(intermediates, "bnbias", bnbias, bnbias_grad)
  return loss, intermediates

def bndiff2_fn(bndiff, Yb):
  jax.debug.print("{}", "In bndiff2")
  bndiff2 = bndiff**2

  def bnvar_fn(bndiff2, Yb):
    jax.debug.print("{}", "In bnvar")
    bnvar = 1/(n-1)*(bndiff2).sum(0, keepdims=True)

    def bnvar_inv_fn(bnvar, Yb):
      jax.debug.print("{}", "In bnvar_inv")
      bnvar_inv = (bnvar + 1e-5)**-0.5
      (loss, intermediates), (bndiff_grad, bnvar_inv_grad) = jax.value_and_grad(bnraw_fn, has_aux=True, argnums=(0,1))(bndiff, bnvar_inv, Yb)
      intermediates = add_value_grad(intermediates, "bndiff", bndiff, bndiff_grad)
      intermediates = add_value_grad(intermediates, "bnvar_inv", bnvar_inv, bnvar_inv_grad)
      return loss, intermediates

    (loss, intermediates), bnvar_grad = jax.value_and_grad(bnvar_inv_fn, has_aux=True)(bnvar, Yb)
    intermediates = add_value_grad(intermediates, "bnvar", bnvar, bnvar_grad)
    return loss, intermediates

  (loss, intermediates), bndiff2_grad = jax.value_and_grad(bnvar_fn, has_aux=True)(bndiff2, Yb)
  intermediates = add_value_grad(intermediates, "bndiff2", bndiff2, bndiff2_grad)
  return loss, intermediates

def bndiff_fn(bmeani, hprebn, Yb):
  jax.debug.print("{}", "In bndiff")
  bndiff = hprebn - bmeani
  (loss, intermediates), bndiff_grad = jax.value_and_grad(bndiff2_fn, has_aux=True)(bndiff, Yb)
  intermediates = add_value_grad(intermediates, "bndiff", bndiff, bndiff_grad)
  return loss, intermediates

def hprebn_fn(embcat, W1, b1, Yb):
  jax.debug.print("{}", "In hprebn")
  hprebn = embcat @ W1 + b1

  def bmeani_fn(hprebn, Yb):
    jax.debug.print("{}", "In bmeani")
    bmeani = 1/n*hprebn.sum(0, keepdims=True)
    (loss, intermediates), (bmeani_grad, hprebn_grad) = jax.value_and_grad(bndiff_fn, has_aux=True, argnums=(0,1))(bmeani, hprebn, Yb)
    intermediates = add_value_grad(intermediates, "bmeani", bmeani, bmeani_grad)
    intermediates = add_value_grad(intermediates, "hprebn", hprebn, hprebn_grad)
    return loss, intermediates

  (loss, intermediates), hprebn_grad = jax.value_and_grad(bmeani_fn, has_aux=True)(hprebn, Yb)
  intermediates = add_value_grad(intermediates, "hprebn", hprebn, hprebn_grad)
  return loss, intermediates

def embcat_fn(emb, Yb):
  jax.debug.print("{}", "In embcat")
  embcat = emb.reshape(emb.shape[0], -1)
  (loss, intermediates), (embcat_grad, W1_grad, b1_grad) = jax.value_and_grad(hprebn_fn, has_aux=True, argnums=(0,1,2))(embcat, W1, b1, Yb)
  intermediates = add_value_grad(intermediates, "embcat", embcat, embcat_grad)
  intermediates = add_value_grad(intermediates, "W1", W1, W1_grad)
  intermediates = add_value_grad(intermediates, "b1", b1, b1_grad)
  return loss, intermediates

def emb_fn(C, Xb, Yb):
  jax.debug.print("{}", "In emb")
  emb = C[Xb]
  (loss, intermediates), emb_grad = jax.value_and_grad(embcat_fn, has_aux=True)(emb, Yb)
  intermediates = add_value_grad(intermediates, "emb", emb, emb_grad)
  return loss, intermediates

def forward(C, Xb, Yb):
  jax.debug.print("{}", "In forward")
  (loss, intermediates), C_grad = jax.value_and_grad(emb_fn, has_aux=True)(C, Xb, Yb)
  intermediates = add_value_grad(intermediates, "C", C, C_grad)
  return loss, intermediates

In [ ]:
loss, intermediates = jax.jit(forward)(C, Xb, Yb)

In forward
In emb
In embcat
In hprebn
In bmeani
In bndiff
In bndiff2
In bnvar
In bnvar_inv
In bnraw
In hpreact
In h
In logits
In logit_maxes
In norm_logits
In counts
In counts_sum
In counts_sum_inv
In probs
In logprobs
In loss


In [ ]:
loss

Array(3.4020648, dtype=float32)

In [ ]:
# Exercise 1: backprop through the whole thing manually,
# backpropagating through exactly all of the variables
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# -----------------
dlogprobs = jnp.zeros_like(logprobs)
dlogprobs = dlogprobs.at[jnp.arange(n), Yb].set(-1/logprobs.shape[0])
cmp('logprobs', dlogprobs, intermediates["logprobs"]["grad"])

dprobs = 1/probs * dlogprobs
cmp('probs', dprobs, intermediates["probs"]["grad"])

dcounts_sum_inv = (counts * dprobs).sum(1, keepdims=True)
dcounts = counts_sum_inv * dprobs
cmp('counts_sum_inv', dcounts_sum_inv, intermediates["counts_sum_inv"]["grad"])

dcounts_sum = -(counts_sum**-2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, intermediates["counts_sum"]["grad"])

dcounts += dcounts_sum
cmp('counts', dcounts, intermediates["counts"]["grad"])

dnorm_logits = jnp.exp(norm_logits) * dcounts
cmp('norm_logits', dnorm_logits, intermediates["norm_logits"]["grad"])

dlogit_maxes = -dnorm_logits.sum(1, keepdims=True)
dlogits = dnorm_logits.clone()
cmp('logit_maxes', dlogit_maxes, intermediates["logit_maxes"]["grad"])

dlogits += jax.nn.one_hot(logits.argmax(1), num_classes=logits.shape[1]) * dlogit_maxes
cmp('logits', dlogits, intermediates["logits"]["grad"])

dh = dlogits @ W2.T
cmp('h', dh, intermediates["h"]["grad"])

dW2 = h.T @ dlogits
cmp('W2', dW2, intermediates["W2"]["grad"])

db2 = dlogits.sum(0)
cmp('b2', db2, intermediates["b2"]["grad"])

dhpreact = (1.0-(h**2)) * dh
cmp('hpreact', dhpreact, intermediates["hpreact"]["grad"])

dbngain = (bnraw * dhpreact).sum(0, keepdims=True)
cmp('bngain', dbngain, intermediates["bngain"]["grad"])

dbnbias = dhpreact.sum(0, keepdims=True)
cmp('bnbias', dbnbias, intermediates["bnbias"]["grad"])

dbnraw = bngain * dhpreact
cmp('bnraw', dbnraw, intermediates["bnraw"]["grad"])

dbnvar_inv = (bndiff * dbnraw).sum(0, keepdims=True)
dbndiff = bnvar_inv * dbnraw
cmp('bnvar_inv', dbnvar_inv, intermediates["bnvar_inv"]["grad"])

dbnvar = - 0.5 * (bnvar+1e-5)**-1.5 * dbnvar_inv
cmp('bnvar', dbnvar, intermediates["bnvar"]["grad"])

dbndiff2 = 1/(n-1) * jnp.ones_like(bndiff2) * dbnvar
cmp('bndiff2', dbndiff2, intermediates["bndiff2"]["grad"])

dbndiff += 2 * bndiff * dbndiff2
cmp('bndiff', dbndiff, intermediates["bndiff"]["grad"])

dbmeani = (-dbndiff).sum(0, keepdims=True)
cmp('bmeani', dbmeani, intermediates["bmeani"]["grad"])

dhprebn = dbndiff.clone()
dhprebn += 1.0/n * (jnp.ones_like(hprebn)) * dbmeani
cmp('hprebn', dhprebn, intermediates["hprebn"]["grad"])

dembcat = dhprebn @ W1.T
cmp('embcat', dembcat, intermediates["embcat"]["grad"])

dW1 = embcat.T @ dhprebn
cmp('W1', dW1, intermediates["W1"]["grad"])

db1 = dhprebn.sum(0)
cmp('b1', db1, intermediates["b1"]["grad"])

demb = dembcat.reshape(emb.shape)
cmp('emb', demb, intermediates["emb"]["grad"])

dC = jax.vmap(jnp.matmul, in_axes=(0, 0))(jax.nn.one_hot(Xb, num_classes=vocab_size).transpose((0,2,1)), demb).sum(0)
cmp('C', dC, intermediates["C"]["grad"])

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10
bngain          | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09
bnbias          | exact: False | approximate: True  | maxdiff: 3.725290298461914e-09
bnraw   

In [ ]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = optax.softmax_cross_entropy(logits, jax.nn.one_hot(Yb, num_classes=vocab_size)).mean()
print(loss_fast.item(), 'diff:', abs((loss_fast - loss).item()))

3.402064800262451 diff: 0.0


In [ ]:
# backward pass

# -----------------
# YOUR CODE HERE :)
dlogits = probs.clone()
dlogits = dlogits.at[jnp.arange(n), Yb].subtract(1)
dlogits /= logits.shape[0]
# -----------------

cmp('logits', dlogits, intermediates["logits"]["grad"])

logits          | exact: False | approximate: True  | maxdiff: 9.080395102500916e-09


In [ ]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdims=True)) / jnp.sqrt(jnp.var(hprebn, 0, keepdims=True, ddof=1) + 1e-5) + bnbias
print('max diff:', abs((hpreact_fast - hpreact)).max())

max diff: 4.7683716e-07


In [ ]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
dhprebn = bngain * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0) - n/(n-1) * bnraw * (dhpreact * bnraw).sum(0))
# -----------------

cmp('hprebn', dhprebn, intermediates["hprebn"]["grad"]) # I can only get approximate to be true, my maxdiff is 9e-10

hprebn          | exact: False | approximate: True  | maxdiff: 1.3969838619232178e-09


In [ ]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g, k1 = jax.random.split(g)
C  = jax.random.normal(k1, (vocab_size, n_embd))
# Layer 1
g, k2, k3 = jax.random.split(g, 3)
W1 = jax.random.normal(k2, (n_embd * block_size, n_hidden)) * (5/3)/((n_embd * block_size)**0.5)
b1 = jax.random.normal(k3, (n_hidden,)) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
g, k4, k5 = jax.random.split(g, 3)
W2 = jax.random.normal(k4, (n_hidden, vocab_size)) * 0.1
b2 = jax.random.normal(k5, vocab_size) * 0.1
# BatchNorm parameters
g, k6, k7 = jax.random.split(g, 3)
bngain = jax.random.normal(k6, (1, n_hidden))*0.1 + 1.0
bnbias = jax.random.normal(k7, (1, n_hidden))*0.1

op = None
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.size for p in parameters)) # number of parameters in total

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

def f(p, X, Y):
  emb = p[0][X] # embed the characters into vectors
  embcat = emb.reshape(emb.shape[0], -1) # concatenate the vectors
  hprebn = embcat @ p[1] + p[2] # hidden layer pre-activation
  # BatchNorm layer
  # -------------------------------------------------------------
  bnmean = hprebn.mean(0, keepdims=True)
  bnvar = hprebn.var(0, keepdims=True, ddof=1)
  bnvar_inv = (bnvar + 1e-5)**-0.5
  bnraw = (hprebn - bnmean) * bnvar_inv
  hpreact = p[5] * bnraw + p[6]
  # -------------------------------------------------------------
  # Non-linearity
  h = jax.nn.tanh(hpreact) # hidden layer
  logits = h @ p[3] + p[4] # output layer
  loss = optax.softmax_cross_entropy(logits, jax.nn.one_hot(Yb, num_classes=vocab_size)).mean()

  return loss

loss_cmp_fn = jax.value_and_grad(jax.jit(f))

4137


In [ ]:
# kick off optimization
for i in range(max_steps):
  # minibatch construct
  g, k8 = jax.random.split(g)
  ix = jax.random.randint(k8, (batch_size,), 0, Xtr.shape[0])
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

  # forward pass
  emb = parameters[0][Xb] # embed the characters into vectors
  embcat = emb.reshape(emb.shape[0], -1) # concatenate the vectors
  # Linear layer
  hprebn = embcat @ parameters[1] + parameters[2] # hidden layer pre-activation
  # BatchNorm layer
  # -------------------------------------------------------------
  bnmean = hprebn.mean(0, keepdims=True)
  bnvar = hprebn.var(0, keepdims=True, ddof=1)
  bnvar_inv = (bnvar + 1e-5)**-0.5
  bnraw = (hprebn - bnmean) * bnvar_inv
  hpreact = parameters[5] * bnraw + parameters[6]
  # -------------------------------------------------------------
  # Non-linearity
  h = jax.nn.tanh(hpreact) # hidden layer
  logits = h @ parameters[3] + parameters[4] # output layer
  loss = optax.softmax_cross_entropy(logits, jax.nn.one_hot(Yb, num_classes=vocab_size)).mean() # loss function

  # loss_cmp, grads_cmp = loss_cmp_fn(parameters, Xb, Yb)

  # manual backprop! #swole_doge_meme
  # -----------------
  # YOUR CODE HERE :)
  counts = jnp.exp(logits - logits.max(axis=1, keepdims=True))
  dlogits = counts/counts.sum(1, keepdims=True)
  dlogits = dlogits.at[jnp.arange(n), Yb].subtract(1)
  dlogits /= logits.shape[0]

  dh = dlogits @ parameters[3].T

  dW2 = h.T @ dlogits

  db2 = dlogits.sum(0)

  dhpreact = (1.0-(h**2)) * dh

  dbngain = (bnraw * dhpreact).sum(0, keepdims=True)
  dbnbias = dhpreact.sum(0, keepdims=True)

  dhprebn = parameters[5] * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0) - n/(n-1) * bnraw * (dhpreact * bnraw).sum(0))

  dembcat = dhprebn @ parameters[1].T

  dW1 = embcat.T @ dhprebn
  db1 = dhprebn.sum(0)

  demb = dembcat.reshape(emb.shape)

  dC = jax.vmap(jnp.matmul, in_axes=(0, 0))(jax.nn.one_hot(Xb, num_classes=vocab_size).transpose((0,2,1)), demb).sum(0)

  grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
  # -----------------

  # update
  lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
  parameters = jax.tree.map(lambda p, gr: p.at[:].subtract(gr * lr), parameters, grads)

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(jnp.log10(loss).item())

  # if i >= 10: # TODO: delete early breaking when you're ready to train the full net
  #   break

      0/ 200000: 3.4393
  10000/ 200000: 2.7880
  20000/ 200000: 1.9147
  30000/ 200000: 2.2150
  40000/ 200000: 2.4352
  50000/ 200000: 2.2320
  60000/ 200000: 2.1384
  70000/ 200000: 2.1175
  80000/ 200000: 2.3172
  90000/ 200000: 2.2111
 100000/ 200000: 1.9375
 110000/ 200000: 2.3230
 120000/ 200000: 2.2682


In [ ]:
C, W1, b1, W2, b2, bngain, bnbias = parameters

In [ ]:
# calibrate the batch norm at the end of training

# pass the training set through
emb = C[Xtr]
embcat = emb.reshape(emb.shape[0], -1)
hpreact = embcat @ W1 + b1
# measure the mean/std over the entire training set
bnmean = hpreact.mean(0, keepdims=True)
bnvar = hpreact.var(0, keepdims=True, ddof=1)

In [ ]:
# evaluate train and val loss

def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.reshape(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = jax.nn.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = optax.softmax_cross_entropy(logits, jax.nn.one_hot(y, num_classes=vocab_size)).mean()
  print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# sample from the model
g, k1 = jax.random.split(g)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[jnp.array([context])] # (1,block_size,d)
      embcat = emb.reshape(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = jax.nn.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = jax.nn.softmax(logits, axis=1)
      ix = jax.random.multinomial(k1, 1, probs).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))